In [0]:
from pyspark.sql.types import StructType,StructField,StringType,IntegerType,DataType,BooleanType
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catelog_name","ecommers","Catalog Name")
dbutils.widgets.text("storage_account_name",'ecommcistorage05',"storage account name")
dbutils.widgets.text("container","ecom-raw-data","Container Name")


In [0]:
catelog_name=dbutils.widgets.get("catelog_name")
storage_account_name=dbutils.widgets.get("storage_account_name")
container=dbutils.widgets.get("container")

In [0]:
adls_path=f"abfss://{container}@{storage_account_name}.dfs.core.windows.net/order_items/landing/"

bronze_checkpoint_path=f"abfss://{container}@{storage_account_name}.dfs.core.windows.net/checkpoint/bronze/fact_order_items/"

In [0]:
spark.readStream \
 .format("cloudFiles") \
 .option("cloudFiles.format", "csv")  \
 .option("cloudFiles.schemaLocation", bronze_checkpoint_path) \
 .option("cloudFiles.schemaEvolutionMode", "rescue") \
 .option("header", "true") \
 .option("cloudFiles.inferColumnTypes", "true") \
 .option("rescuedDataColumn", "_rescued_data") \
 .option("cloudFiles.includeExistingFiles", "true")  \
 .option("pathGlobFilter", "*.csv") \
 .load(adls_path) \
 .withColumn("ingest_timestamp", F.current_timestamp()) \
 .withColumn("source_file", F.col("_metadata.file_path")) \
 .writeStream \
 .outputMode("append") \
 .option("checkpointLocation", bronze_checkpoint_path) \
 .trigger(availableNow=True) \
 .toTable(f"{catelog_name}.bronze.brz_order_items") \
 .awaitTermination()

In [0]:
display(spark.sql(f"SELECT max(dt) FROM {catelog_name}.bronze.brz_order_items"))
display(spark.sql(f"SELECT min(dt) FROM {catelog_name}.bronze.brz_order_items"))

In [0]:
%sql
SELECT count(*) FROM cloud_files_state('abfss://ecomm-raw-data@datalakeindia.dfs.core.windows.net/checkpoint/bronze/fact_order_items/')